# Embed `chunks_llm.json` → upsert Qdrant (Colab GPU)

Embed trường `embedding_text` bằng `AITeamVN/Vietnamese_Embedding` (BGE-M3, 1024-dim) rồi upsert thẳng vào Qdrant remote.

**Trước khi chạy:**
1. Runtime → Change runtime type → chọn GPU (L4 / A100).
2. Upload `chunks_llm.json` vào Files panel (hoặc mount Google Drive).
3. Điền `QDRANT_API_KEY` ở Cell *Config* (lấy từ root `.env`).

**Đặc điểm:** không dùng `asyncio.run` (kẹt event loop của Jupyter); resumable qua `embed_progress.json`; point id = `uuid5(NS, chunk_id)` khớp `app/core/qdrant.py`.

## 1. Cài thư viện

In [ ]:
!pip install -q sentence-transformers qdrant-client

## 2. Config — điền vào đây

In [ ]:
QDRANT_HOST    = "114.29.239.135"   # public IP remote server
QDRANT_PORT    = 6333
QDRANT_API_KEY = "YOUR_API_KEY"     # lấy từ root .env -> QDRANT_API_KEY
COLLECTION     = "history_vn_chunks"
CHUNKS_FILE    = "chunks_llm.json"      # upload file này lên Colab trước
PROGRESS_FILE  = "embed_progress.json"  # tự tạo, dùng để resume
BATCH_SIZE     = 128   # A100: thử 256 nếu VRAM còn nhiều
EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding"

## 3. Import + helpers (copy từ `app/core/qdrant.py`)

In [ ]:
import json, uuid, re, os
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Namespace cố định — phải khớp app/core/qdrant.py::_NS
_NS = uuid.UUID("a3f1c2e4-5b6d-4e8a-9c0f-1d2e3f4a5b6c")

def point_id_for(chunk_id: str) -> str:
    return str(uuid.uuid5(_NS, chunk_id))

_HEADING_KEY_RE = re.compile(r"^h(\d+)$", re.IGNORECASE)

def build_heading_path(metadata: dict) -> list:
    headings = metadata.get("headings")
    if not isinstance(headings, dict):
        return []
    ordered = []
    for key, value in headings.items():
        m = _HEADING_KEY_RE.match(str(key))
        text = str(value).strip() if value else ""
        if m and text:
            ordered.append((int(m.group(1)), text))
    return [t for _, t in sorted(ordered)]

def _str_list(value) -> list:
    if not isinstance(value, list):
        return []
    return [str(i).strip() for i in value if str(i).strip()]

def make_payload(raw: dict) -> dict:
    meta = raw.get("metadata") or {}
    return {
        "chunk_id":     raw["chunk_id"],
        "heading_path": build_heading_path(meta),
        "events":       _str_list(meta.get("events")),
        "actors":       _str_list(meta.get("actors")),
        "times":        _str_list(meta.get("times")),
        "locations":    _str_list(meta.get("locations")),
    }

## 4. Load chunks

In [ ]:
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

print(f"Loaded {len(raw_chunks)} chunks")
print(f"Sample chunk_id: {raw_chunks[0]['chunk_id']}")

## 5. Kết nối Qdrant + tạo collection nếu chưa có

In [ ]:
client = QdrantClient(
    host=QDRANT_HOST,
    port=QDRANT_PORT,
    api_key=QDRANT_API_KEY,
    https=False,      # server chạy HTTP thuần, KHÔNG HTTPS
    timeout=60,
)

if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print(f"Created collection '{COLLECTION}'")
else:
    info = client.get_collection(COLLECTION)
    print(f"Collection '{COLLECTION}' exists — {info.points_count} points hiện có")

## 6. Load model (lần đầu ~30s download + load lên GPU)

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")
model.max_seq_length = 2048
print(f"Model loaded on: {model.device}")

## 7. Embed + upsert (resumable)

In [ ]:
# Load progress
progress = set()
if Path(PROGRESS_FILE).exists():
    with open(PROGRESS_FILE, "r") as f:
        progress = set(json.load(f))
    print(f"Resume: {len(progress)} chunk đã upsert trước đó")

todo = [c for c in raw_chunks if c["chunk_id"] not in progress]
print(f"Cần upsert: {len(todo)} chunk")

total_upserted = len(progress)
for start in range(0, len(todo), BATCH_SIZE):
    batch = todo[start : start + BATCH_SIZE]
    texts = [str(c.get("embedding_text") or c["text"]) for c in batch]

    # Embed trên GPU (sync, không cần asyncio)
    vectors = model.encode(
        texts,
        normalize_embeddings=True,   # L2 normalize → cosine = dot product
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    points = [
        PointStruct(
            id=point_id_for(c["chunk_id"]),
            vector=vec.tolist(),
            payload=make_payload(c),
        )
        for c, vec in zip(batch, vectors)
    ]

    client.upsert(collection_name=COLLECTION, points=points, wait=True)

    for c in batch:
        progress.add(c["chunk_id"])
    total_upserted += len(batch)

    with open(PROGRESS_FILE, "w") as f:
        json.dump(list(progress), f)

    pct = total_upserted / len(raw_chunks) * 100
    print(f"[{total_upserted}/{len(raw_chunks)} — {pct:.1f}%] batch {start // BATCH_SIZE + 1}")

print(f"\nDone! Total upserted: {total_upserted}")

## 8. Verify

In [ ]:
info = client.get_collection(COLLECTION)
print(f"Collection '{COLLECTION}': {info.points_count} points")
assert info.points_count == len(raw_chunks), \
    f"Mismatch: Qdrant có {info.points_count}, chunks_llm.json có {len(raw_chunks)}"
print("OK — point count khớp chunks_llm.json")

---
# Merge knowledge graph → Neo4j

Đọc `graph_extractions.json` (cache entity/quan hệ đã trích) rồi merge vào Neo4j remote. **Không gọi LLM** — chỉ đẩy dữ liệu đã có lên graph.

Logic merge sao y `app/tools/graph_rag/graph_store.py`: 1 label `:Entity` (key `name` UNIQUE) + 1 rel-type `:REL` (phân biệt bằng `keyword`), idempotent, tích lũy `descriptions` + `source_chunk_ids` xuyên chunk.

**Trước khi chạy:** upload `graph_extractions.json` lên Files; điền creds Neo4j ở cell Config bên dưới.

## 9. Cài driver + config Neo4j

In [ ]:
!pip install -q neo4j

NEO4J_URI      = "bolt://114.29.239.135:7687"   # khớp NEO4J_URI trong root .env
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "YOUR_NEO4J_PASSWORD"          # lấy từ root .env -> NEO4J_PASSWORD
GRAPH_FILE     = "graph_extractions.json"       # upload file này lên Colab trước

## 10. Load cache graph

In [ ]:
with open(GRAPH_FILE, "r", encoding="utf-8") as f:
    graph_cache = json.load(f)

n_ent_total = sum(len(v.get("entities", [])) for v in graph_cache.values())
n_rel_total = sum(len(v.get("relations", [])) for v in graph_cache.values())
print(f"Loaded {len(graph_cache)} chunks — {n_ent_total} entities, {n_rel_total} relations")

## 11. Kết nối Neo4j + tạo constraint/index

In [ ]:
from neo4j import GraphDatabase

neo_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
neo_driver.verify_connectivity()

CONSTRAINTS = [
    "CREATE CONSTRAINT entity_name_unique IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE",
    "CREATE INDEX entity_type_idx IF NOT EXISTS FOR (e:Entity) ON (e.type)",
]
with neo_driver.session() as s:
    for c in CONSTRAINTS:
        s.run(c)
print("Neo4j connected — constraint + index sẵn sàng")

## 12. Merge entity + relation (idempotent)

In [ ]:
# Cypher sao y app/tools/graph_rag/graph_store.py
MERGE_ENTITIES = """
UNWIND $entities AS ent
MERGE (e:Entity {name: ent.name})
ON CREATE SET e.type = ent.type,
              e.descriptions = [ent.description],
              e.source_chunk_ids = [$chunk_id]
ON MATCH SET e.type = coalesce(e.type, ent.type),
             e.descriptions = CASE WHEN ent.description IN e.descriptions
                                   THEN e.descriptions ELSE e.descriptions + ent.description END,
             e.source_chunk_ids = CASE WHEN $chunk_id IN e.source_chunk_ids
                                       THEN e.source_chunk_ids ELSE e.source_chunk_ids + $chunk_id END
"""

MERGE_RELATIONS = """
UNWIND $relations AS rel
MATCH (s:Entity {name: rel.source})
MATCH (t:Entity {name: rel.target})
MERGE (s)-[r:REL {keyword: rel.keyword}]->(t)
ON CREATE SET r.descriptions = [rel.description],
              r.source_chunk_ids = [$chunk_id]
ON MATCH SET r.descriptions = CASE WHEN rel.description IN r.descriptions
                                   THEN r.descriptions ELSE r.descriptions + rel.description END,
             r.source_chunk_ids = CASE WHEN $chunk_id IN r.source_chunk_ids
                                       THEN r.source_chunk_ids ELSE r.source_chunk_ids + $chunk_id END
"""

def merge_tx(tx, chunk_id, entities, relations):
    if entities:
        tx.run(MERGE_ENTITIES, entities=entities, chunk_id=chunk_id)
    if relations:
        tx.run(MERGE_RELATIONS, relations=relations, chunk_id=chunk_id)

n_ent = n_rel = n_chunk = 0
with neo_driver.session() as session:
    for cid, entry in graph_cache.items():
        entities = entry.get("entities", [])
        relations = entry.get("relations", [])
        if not entities and not relations:
            continue
        # entity rồi relation trong MỘT transaction để endpoint chắc chắn tồn tại
        session.execute_write(merge_tx, cid, entities, relations)
        n_ent += len(entities)
        n_rel += len(relations)
        n_chunk += 1
        if n_chunk % 100 == 0:
            print(f"merged {n_chunk}/{len(graph_cache)} chunks...")

print(f"\nDone! Merge {n_chunk} chunks — {n_ent} entity, {n_rel} relation (trước dedup)")

## 13. Verify Neo4j

In [ ]:
with neo_driver.session() as s:
    ne = s.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
    nr = s.run("MATCH ()-[r:REL]->() RETURN count(r) AS c").single()["c"]
    sample = s.run("MATCH (e:Entity) RETURN e.name AS name, e.type AS type LIMIT 5").data()
print(f":Entity nodes (sau dedup theo name) = {ne}")
print(f":REL relationships = {nr}")
print("Sample entities:", sample)
neo_driver.close()